In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# =========================
# Load CSV
# =========================
df = pd.read_csv("Swing_Trial.csv")
df.columns = [c.strip() for c in df.columns]

required_cols = ["timestamp_ms", "qw", "qx", "qy", "qz"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

for col in required_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required_cols).reset_index(drop=True)

if len(df) == 0:
    raise ValueError("No valid rows found in Swing_Trial.csv")


# =========================
# Quaternion helper functions
# =========================
def normalize_quat(q):
    q = np.array(q, dtype=float)
    norm = np.linalg.norm(q)
    if norm == 0:
        return np.array([1.0, 0.0, 0.0, 0.0])
    return q / norm

def quat_conj(q):
    w, x, y, z = q
    return np.array([w, -x, -y, -z])

def quat_inv(q):
    return quat_conj(q) / np.dot(q, q)

def quat_mult(q1, q2):
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    return np.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2
    ])

def quat_to_rotmat(q):
    w, x, y, z = q
    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - z*w),     2*(x*z + y*w)],
        [2*(x*y + z*w),     1 - 2*(x*x + z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w),     2*(y*z + x*w),     1 - 2*(x*x + y*y)]
    ])


# =========================
# Build baseline-corrected quaternions
# First 2 seconds = address baseline
# =========================
df["time_s"] = (df["timestamp_ms"] - df["timestamp_ms"].iloc[0]) / 1000.0
baseline_df = df[df["time_s"] <= 2.0].copy()

if len(baseline_df) == 0:
    raise ValueError("No baseline data found in first 2 seconds.")

q0 = normalize_quat([
    baseline_df["qw"].mean(),
    baseline_df["qx"].mean(),
    baseline_df["qy"].mean(),
    baseline_df["qz"].mean()
])

relative_quats = []
for _, row in df.iterrows():
    q = normalize_quat([row["qw"], row["qx"], row["qy"], row["qz"]])
    q_rel = quat_mult(quat_inv(q0), q)
    q_rel = normalize_quat(q_rel)
    relative_quats.append(q_rel)

relative_quats = np.array(relative_quats)
df["rw"] = relative_quats[:, 0]
df["rx"] = relative_quats[:, 1]
df["ry"] = relative_quats[:, 2]
df["rz"] = relative_quats[:, 3]


# =========================
# Use only motion portion if gyro exists
# Otherwise animate full file
# =========================
if all(col in df.columns for col in ["gx", "gy", "gz"]):
    for col in ["gx", "gy", "gz"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["gx", "gy", "gz"]).reset_index(drop=True)
    df["gyro_mag"] = np.sqrt(df["gx"]**2 + df["gy"]**2 + df["gz"]**2)

    max_gyro = df["gyro_mag"].max()
    threshold = max(0.05, 0.15 * max_gyro)

    moving_indices = df.index[df["gyro_mag"] > threshold]

    if len(moving_indices) > 0:
        start_idx = max(0, moving_indices[0] - 10)
        end_idx = min(len(df) - 1, moving_indices[-1] + 10)
        df_anim = df.iloc[start_idx:end_idx + 1].copy().reset_index(drop=True)
    else:
        df_anim = df.copy()
else:
    df_anim = df.copy()


# =========================
# Compute rotated forearm tip positions
# =========================
FOREARM_LENGTH = 10.0
base_vector = np.array([0.0, 0.0, FOREARM_LENGTH])

tips = []
for _, row in df_anim.iterrows():
    q = normalize_quat([row["rw"], row["rx"], row["ry"], row["rz"]])
    R = quat_to_rotmat(q)
    tip = R @ base_vector
    tips.append(tip)

tips = np.array(tips)

print("First 5 tip vectors:")
print(tips[:5])


# =========================
# Test one static frame first
# =========================
test_idx = min(10, len(tips) - 1)
tip = tips[test_idx]

fig_test = plt.figure(figsize=(6, 6))
ax_test = fig_test.add_subplot(111, projection="3d")
ax_test.plot([0, tip[0]], [0, tip[1]], [0, tip[2]], lw=3)
ax_test.set_xlim(-12, 12)
ax_test.set_ylim(-12, 12)
ax_test.set_zlim(-12, 12)
ax_test.set_xlabel("X")
ax_test.set_ylabel("Y")
ax_test.set_zlabel("Z")
ax_test.set_title("Static Test Frame")
plt.tight_layout()
plt.show()


# =========================
# 3D Animation
# =========================
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")

def setup_axes():
    ax.set_xlim(-12, 12)
    ax.set_ylim(-12, 12)
    ax.set_zlim(-12, 12)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title("3D Wrist/Forearm Orientation")

setup_axes()

# Forearm line
line, = ax.plot([], [], [], lw=4)

# Trail of the forearm tip
trail, = ax.plot([], [], [], alpha=0.5)

# Origin point
origin, = ax.plot([0], [0], [0], marker="o")

def init():
    line.set_data([], [])
    line.set_3d_properties([])
    trail.set_data([], [])
    trail.set_3d_properties([])
    return line, trail, origin

def update(frame):
    tip = tips[frame]

    # Update forearm line from origin to tip
    line.set_data([0, tip[0]], [0, tip[1]])
    line.set_3d_properties([0, tip[2]])

    # Update trail
    trail.set_data(tips[:frame + 1, 0], tips[:frame + 1, 1])
    trail.set_3d_properties(tips[:frame + 1, 2])

    return line, trail, origin

ani = FuncAnimation(
    fig,
    update,
    frames=len(tips),
    init_func=init,
    interval=40,
    blit=False,
    repeat=True
)

plt.tight_layout()
plt.show()

# Optional save line if you install ffmpeg:
# ani.save("swing_3d.mp4", fps=25)

ValueError: Missing required column: timestamp_ms